In [1]:
import pandas as pd

df = pd.read_excel("../data/raw/Online Retail.xlsx", engine="openpyxl")

# 1. Drop rows with missing InvoiceNo
df = df[df['InvoiceNo'].notna()]

# 2. Remove rows where UnitPrice <= 0
df = df[df['UnitPrice'] > 0]

# 3. Tag returns
df['IsReturn'] = df['Quantity'] < 0

# 4. Clean description
df['Description'] = df['Description'].str.strip().str.lower()

# 5. Add revenue
df['Revenue'] = df['Quantity'] * df['UnitPrice']

df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn,Revenue
0,536365,85123A,white hanging heart t-light holder,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,False,15.30
1,536365,71053,white metal lantern,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
2,536365,84406B,cream cupid hearts coat hanger,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,False,22.00
3,536365,84029G,knitted union flag hot water bottle,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34
4,536365,84029E,red woolly hottie white heart.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,False,20.34


In [2]:
df.to_csv("../data/processed/online_retail_clean_step1.csv", index=False)

### Finding suspicious / non-product StockCodes

In [5]:
# show top 50 unique stockcodes (quick peek)
print(df['StockCode'].unique()[:50])

# show stockcodes that contain letters, with counts — top 50
mask = df['StockCode'].str.contains('[A-Za-z]', regex=True, na=False)
letter_counts = df.loc[mask, 'StockCode'].value_counts()
display(letter_counts.head(50))

# show stockcodes whose description contains 'post' or looks like postage
df[df['Description'].str.contains('post|postage|delivery', na=False)]['StockCode'].value_counts().head(20)

['85123A' 71053 '84406B' '84029G' '84029E' 22752 21730 22633 22632 84879
 22745 22748 22749 22310 84969 22623 22622 21754 21755 21777 48187 22960
 22913 22912 22914 21756 22728 22727 22726 21724 21883 10002 21791 21035
 22326 22629 22659 22631 22661 21731 22900 21913 22540 22544 22492 'POST'
 22086 20679 37370 21871]


StockCode
85123A     2307
85099B     2156
POST       1252
85099C      960
82494L      938
85099F      837
DOT         707
M           565
84970S      546
84596B      541
84997D      481
84029G      473
15056N      467
84970L      460
84596F      452
84029E      449
47591D      445
85049E      423
47590B      401
47590A      394
84997B      377
47566B      376
85014B      355
84030E      346
84536A      346
84997C      338
85049A      332
15056BL     326
84032A      318
47559B      294
84406B      293
85014A      280
85049G      277
85199S      275
84997A      263
84032B      262
16161P      250
84596G      249
47593B      247
84510A      247
51014A      244
35471D      236
84509A      233
85049C      231
75049L      231
47504K      218
85132C      216
85131D      206
85184C      197
48173C      194
Name: count, dtype: int64

StockCode
POST     1252
DOT       707
21135     186
23394     155
85015     145
22944     116
23550      64
21459      21
21460      11
21461       9
23620       7
21769       2
Name: count, dtype: int64

### Creating a blacklist and filter them out

In [6]:
# example blacklist (update with what you found)
blacklist = ['POST', 'DOT', 'M', 'BANK CHARGES', 'SAMPLES', 'D']  # replace/extend

# keep rows whose StockCode is NOT in blacklist
df2 = df[~df['StockCode'].isin(blacklist)].copy()

# quick checks
print("Rows before:", len(df))
print("Rows after:", len(df2))
print("Removed rows:", len(df) - len(df2))

Rows before: 539392
Rows after: 536754
Removed rows: 2638


### Verifying returns vs cancellation invoices consistency

In [7]:
# negative qty rows not in C invoices
neg_not_C = df2[(df2['Quantity'] < 0) & (~df2['InvoiceNo'].astype(str).str.startswith('C'))]
print("Negative qty but Invoice not starting with C:", len(neg_not_C))

# C invoices without negative qty
C_not_neg = df2[(df2['InvoiceNo'].astype(str).str.startswith('C')) & (df2['Quantity'] >= 0)]
print("Invoice startswith 'C' but Quantity not negative:", len(C_not_neg))

# small sample to inspect
display(neg_not_C.head())
display(C_not_neg.head())

Negative qty but Invoice not starting with C: 0
Invoice startswith 'C' but Quantity not negative: 0


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn,Revenue


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsReturn,Revenue


### Deciding treatment of returns for revenue

In [8]:
# revenue excluding returns
df_pos = df2[df2['Quantity'] > 0].copy()
total_revenue_all = df2['Revenue'].sum()
total_revenue_pos = df_pos['Revenue'].sum()
print("Total revenue (all rows):", total_revenue_all)
print("Total revenue (excluding returns):", total_revenue_pos)

Total revenue (all rows): 9578941.983000001
Total revenue (excluding returns): 10304058.623


### Final clean and export

In [9]:
# final cleaned CSV
df2.to_csv("../data/processed/online_retail_clean_final.csv", index=False)
print("Saved ../data/processed/online_retail_clean_final.csv")

Saved ../data/processed/online_retail_clean_final.csv
